In [48]:
#base
import kagglehub
import pandas as pd
import numpy as np

#1
from sklearn.neighbors import KNeighborsClassifier
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

#2, 6
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Lasso, Ridge

#3
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

#5
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC, NuSVC

#6
from sklearn import preprocessing as skpre
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error

#7
from sklearn import model_selection as skms
from sklearn.model_selection import GridSearchCV

#8
from sklearn import ensemble
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

#9
from sklearn import svm
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

#10
import eli5
from eli5.sklearn import PermutationImportance
from sklearn.ensemble import RandomForestClassifier
from IPython.display import display # (interface) - wrapper za prikaz tablica i sl.

#dodatan import koji uklanja upozorenja zbog bolje preglednosti rješenja
import warnings
warnings.filterwarnings('ignore')

In [44]:
!pip install -U eli5
#upgrade eli5 - iz nekog razloga se nije prikazivala tablic "weights", no upgrade eli5 nije rješilo problem.
#iz tog razloga se koristi IPython.display

In [4]:
path = kagglehub.dataset_download("l3llff/banana")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'banana' dataset.
Path to dataset files: /kaggle/input/banana


In [5]:
df = pd.read_csv("/kaggle/input/banana/banana_quality.csv")
df.head()

,Size,Weight,Sweetness,Softness,HarvestTime,Ripeness,Acidity,Quality
0,-1.924968,0.468078,3.077832,-1.472177,0.294799,2.435570,0.271290,Good
1,-2.409751,0.486870,0.346921,-2.495099,-0.892213,2.067549,0.307325,Good
2,-0.357607,1.483176,1.568452,-2.645145,-0.647267,3.090643,1.427322,Good
3,-0.868524,1.566201,1.889605,-1.273761,-1.006278,1.873001,0.477862,Good
4,0.651825,1.319199,-0.022459,-1.209709,-1.430692,1.078345,2.812442,Good


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Size         8000 non-null   float64
 1   Weight       8000 non-null   float64
 2   Sweetness    8000 non-null   float64
 3   Softness     8000 non-null   float64
 4   HarvestTime  8000 non-null   float64
 5   Ripeness     8000 non-null   float64
 6   Acidity      8000 non-null   float64
 7   Quality      8000 non-null   object 
dtypes: float64(7), object(1)
memory usage: 500.1+ KB


#1. ZADATAK

Proučite dataset i definirajte što je target varijabla. Ako je ne pronalazite, izaberite neku varijablu za koju postoje samo dvije vrijednosti u stupcu. Iz dataseta izbacite sve značajke koje nisu numeričke, a ako negdje nedostaje vrijednost umetnite medijan. 20% dataseta ostavite za testiranje, a pri dijeljenju dataset-a u funkciji train_test_split koristite random_state=42 i stratify=y. Istrenirajte k-NN sa 5 susjeda i ispišite koja je točnost modela. Istrenirajte i model naive bayes i ispišite točnost.

In [7]:
y = df["Quality"]
X = df.drop(["Quality"], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

#knn
knn_1 = KNeighborsClassifier(n_neighbors=5)
knn_1.fit(X_train, y_train)
pred_knn_1 = knn_1.predict(X_test)
acc_knn = metrics.accuracy_score(y_test, pred_knn_1)
print(f'Accuracy KNN: {acc_knn*100:.2f}%')

#Naive Bayes
gnb = GaussianNB()
gnb.fit(X_train, y_train)
pred_gnb = gnb.predict(X_test)
acc_gnb = metrics.accuracy_score(y_test, pred_gnb)
print(f'Accuracy GNB: {acc_gnb*100:.2f}%')

Accuracy KNN: 97.81%
Accuracy GNB: 88.56%


*Zaključak: KNN je bolji model na ovom datasetu.*

#2. ZADATAK
Proučite dataset i definirajte što je target varijabla. Iz dataseta izbacite sve značajke koje nisu numeričke, a ako negdje nedostaje vrijednost umetnite medijan. 20% dataseta ostavite za testiranje, a pri dijeljenju dataset-a u funkciji train_test_split koristite random_state=42. Istrenirajte k-NN regresiju sa 3 susjeda i ispišite koja je srednja kvadratna pogreška. Isto napravite još jednom, ali koristite linearnu regresiju.

In [8]:
y = np.where(df["Quality"] == "Good", 1, 0)
X = df.drop(["Quality"], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

#knn
knn_2 = KNeighborsRegressor(n_neighbors=3)
knn_2.fit(X_train, y_train)
y_pred = knn_2.predict(X_test)

mse = metrics.mean_squared_error(y_test, y_pred)
print(f'KNN MSE: {mse:.4f}')

#lr
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

mse_lr = metrics.mean_squared_error(y_test, y_pred_lr)
print(f'Linear Regression MSE: {mse_lr:.4f}')


KNN MSE: 0.0190
Linear Regression MSE: 0.1101


*Zaključak: S obzirom da manja vrijednost MSE i RMSE znači bolji model, u ovom slučaju KNN je bolji model za navedeni dataset.*

#3. ZADATAK
### Koristite dataset sa linka koji ste preuzeli prethodno, a link je u datoteci datasets_klasifikacija_p6.txt. Model neka bude k-NN sa 5 susjeda. Koristite peterostruku unakrsnu provjeru modela StratifiedKFold sa parametrima n_splits=5, shuffle=True, random_state=42, a točnost dobijte korištenjem cross_val_score uz parametar scoring="accuracy". Prikažite točnosti i prosječnu vrijednost.

In [9]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

knn_3 = KNeighborsClassifier(n_neighbors=5)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
score = cross_val_score(knn_3, X_train, y_train, cv=skf, scoring="accuracy")
print(f'Accuracy: {score}')
print(f'Mean accuracy: {score.mean():.4f}')


Accuracy: [0.97734375 0.9828125  0.97734375 0.97890625 0.97421875]
Mean accuracy: 0.9781


*Zaključak: Točnost po foldovima je poprilično ujednačena te možemo zaključiti kako navedeni model dobro generalizira podatke.*

#4. ZADATAK
Koristite dataset sa linka koji ste preuzeli prethodno, a link je u datoteci datasets_klasifikacija_p6.txt. 20% dataseta ostavite za testiranje, a pri dijeljenju dataset-a u funkciji train_test_split koristite random_state=42 i stratify=y. Istrenirajte k-NN sa 5 susjeda i ispišite sve pokazatelje modela uključujući matricu konfuzije.
Nakon toga  smanjite threshhold za 50%, pa ponovo izračunajte i ispišite sve pokazatelje modela uključujući matricu konfuzije.
Nakon toga  povećajte threshhold za 50% od početnog, pa ponovo izračunajte i ispišite sve pokazatelje modela uključujući matricu konfuzije

In [10]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

knn_4 = KNeighborsClassifier(n_neighbors=5)
knn_4.fit(X_train, y_train)
probs = knn_4.predict_proba(X_test)[:, 1]

# default
print("Threshold default")
print("--------------------------------------------------------------------")
y_pred_4 = (probs >= 0.5).astype(int)
print(metrics.classification_report(y_test, y_pred_4))
print(metrics.confusion_matrix(y_test, y_pred_4))
print("____________________________________________________________________")
print()

# -50%
print("Threshold -50%")
print("--------------------------------------------------------------------")
y_pred_4_25 = (probs >= 0.25).astype(int)
print(metrics.classification_report(y_test, y_pred_4_25))
print(metrics.confusion_matrix(y_test, y_pred_4_25))
print("____________________________________________________________________")
print()

# +50%
print("Threshold +50%")
print("--------------------------------------------------------------------")
y_pred_4_75 =(probs >= 0.75).astype(int)
print(metrics.classification_report(y_test, y_pred_4_75))
print(metrics.confusion_matrix(y_test, y_pred_4_75))

Threshold default
--------------------------------------------------------------------
              precision    recall  f1-score   support

           0       0.97      0.98      0.98       799
           1       0.98      0.97      0.98       801

    accuracy                           0.98      1600
   macro avg       0.98      0.98      0.98      1600
weighted avg       0.98      0.98      0.98      1600

[[786  13]
 [ 22 779]]
____________________________________________________________________

Threshold -50%
--------------------------------------------------------------------
              precision    recall  f1-score   support

           0       0.99      0.96      0.98       799
           1       0.96      0.99      0.98       801

    accuracy                           0.98      1600
   macro avg       0.98      0.98      0.98      1600
weighted avg       0.98      0.98      0.98      1600

[[770  29]
 [ 10 791]]
___________________________________________________________

*Zaključak Threshold -50%*
* *Klasa 0 (dobre banane)*
  * *Precision 0.99: model kaže da je banana dobra, u 99% slučajeva je stvarno dobra.*
  * *Recall 0.96: Od svih stvarno dobrih banana, model točno prepozna 96%, a 4% ih pogrešno označi kao loše.*
  * *F1-score 0.98: Model ima vrlo dobar balans između preciznosti i pokrivenosti za dobre banane.*

* *Klasa 1 (loše banane)*
  * *Precision 0.96: model kaže da je banana loša, u 96% slučajeva je stvarno loša.*
  * *Recall 0.99: Od svih stvarno loših banana, model otkrije 99%, a 1% mu promakne kao dobre.*
  * *F1-score 0.98: I ovdje je balans gotovo savršen.*

#5. ZADATAK
Koristite dataset sa linka koji ste preuzeli prethodno, a link je u datoteci datasets_klasifikacija_p6.txt. 20% dataseta ostavite za testiranje, a pri dijeljenju dataset-a u funkciji train_test_split koristite random_state=42 i stratify=y. Istrenirajte pet klasifikatora (diskriminantna analiza, stablo odlučivanja, SVC, NuSVC, logistička regresija) sa hiperparametrima po želji i ispišite sve pokazatelje svakog modela uključujući matricu konfuzije.

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

"""
tree = DecisionTreeClassifier(max_depth=5, min_samples_split=10, random_state=42)
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)
print('Decision Tree:')
print(f'Classification report: \n {metrics.classification_report(y_test, y_pred_tree)}')
print(f'Decision Tree MC \n {metrics.confusion_matrix(y_test, y_pred_tree)}')
print("____________________________________________________________________")
print()


lda = LinearDiscriminantAnalysis(solver='svd')
lda.fit(X_train, y_train)
y_pred_lda = lda.predict(X_test)
print('Linear Discriminant Analysis:')
print(f'Classification report: \n {metrics.classification_report(y_test, y_pred_lda)}')
print(f'LDA MC \n {metrics.confusion_matrix(y_test, y_pred_lda)}')
print("____________________________________________________________________")
print()


svc = SVC(kernel='linear', C=1, random_state=42, probability=True)
svc.fit(X_train, y_train)
y_pred_svc = svc.predict(X_test)
print('SVC')
print(f'Classification report: \n {metrics.classification_report(y_test, y_pred_svc)}')
print(f'SVC MC \n {metrics.confusion_matrix(y_test, y_pred_svc)}')
print("____________________________________________________________________")
print()


nusvc = NuSVC(kernel='linear', nu=0.1, random_state=42, probability=True)
nusvc.fit(X_train, y_train)
y_pred_nusvc = nusvc.predict(X_test)
print('NuSVC')
print(f'Classification report: \n {metrics.classification_report(y_test, y_pred_nusvc)}')
print(f'NuSVC MC \n {metrics.confusion_matrix(y_test, y_pred_nusvc)}')
print("____________________________________________________________________")
print()


lr_5 = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs')
lr_5.fit(X_train, y_train)
y_pred_lr_5 = lr_5.predict(X_test)
print('Logistic Regression')
print(f'Classification report: \n {metrics.classification_report(y_test, y_pred_lr_5)}')
print(f'Logistic Regression MC \n {metrics.confusion_matrix(y_test, y_pred_lr_5)}')
print("____________________________________________________________________")
print()

"""
# Kraća verzija
models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=5, min_samples_split=10, random_state=42),
    "LDA": LinearDiscriminantAnalysis(solver='svd'),
    "SVC": SVC(kernel='linear', C=1, random_state=42),
    "NuSVC": NuSVC(kernel='linear', nu=0.1, random_state=42),
    "Logistic Regression": LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs')
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"{name}:")
    print(f"Classification report:\n{metrics.classification_report(y_test, y_pred)}")
    print(f"Confusion matrix:\n{metrics.confusion_matrix(y_test, y_pred)}")
    print("____________________________________________________________________\n")


Decision Tree:
Classification report:
              precision    recall  f1-score   support

           0       0.90      0.90      0.90       799
           1       0.90      0.90      0.90       801

    accuracy                           0.90      1600
   macro avg       0.90      0.90      0.90      1600
weighted avg       0.90      0.90      0.90      1600

Confusion matrix:
[[717  82]
 [ 82 719]]
____________________________________________________________________

LDA:
Classification report:
              precision    recall  f1-score   support

           0       0.90      0.85      0.87       799
           1       0.86      0.90      0.88       801

    accuracy                           0.88      1600
   macro avg       0.88      0.87      0.87      1600
weighted avg       0.88      0.88      0.87      1600

Confusion matrix:
[[677 122]
 [ 78 723]]
____________________________________________________________________

SVC:
Classification report:
              precision    rec

*Zaključak: prema podacima Decision Tree je najbolji model, s tim da su ostali modeli vrlo blizu Decision Tree. Decision Tree ima podjednaku precision za obje klase za razliku od ostalih modela gdje su vidljive manje razlike između precision klasa.*

#Priprema dataseta za 6. zadatak

In [13]:
path = kagglehub.dataset_download("harshsingh2209/medical-insurance-payout")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'medical-insurance-payout' dataset.
Path to dataset files: /kaggle/input/medical-insurance-payout


In [14]:
df2 = pd.read_csv("/kaggle/input/medical-insurance-payout/expenses.csv")
df2.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [15]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


#6. ZADATAK
Koristite dataset sa linka koji ste preuzeli prethodno, a link je u datoteci datasets_regresija_p4.txt. Ako nema niti jedne kategorijske značajke, onda jednu numerički pretvorite u kategorijsku. Primjer koda je u nastavku:

import pandas as pd
import numpy as np

df = pd.DataFrame({'income': [1500, 1800, 2200, 2500, 3200, 4000]})
threshold = df['income'].median()  # prag razdvajanja
df['income_category'] = np.where(df['income'] < threshold, 'niska', 'visoka')
print(df)

Tu jednu kategorijsku značajku konvertirajte u numeričke korištenjem OneHotEncoder-a. Model trenirajte sa linearnom regresijom, lasso regresijom i ridge regresijom. Dataset pripremite sa train_test_split funkcijom i pri tom koristite random_state=21, a 20% neka vam ostane za testiranje. Izračunajte RMSE sa testnim podacima za sva tri modela. Ispišite izračunato.

In [16]:
df2_6 = df2.drop(columns=["sex", "smoker", "region"]) #novi dataframe za 6. zadatak

In [17]:
y = df2_6["charges"]
cat_col = ['age']
num_cols = ['bmi', 'children']
X = df2_6[num_cols + cat_col]


pre = ColumnTransformer(
    transformers=[
        ('cat', skpre.OneHotEncoder(drop='first', handle_unknown='ignore'), cat_col),
        ('num', 'passthrough', num_cols)
    ]
)


models = {
    "Linear Regression": LinearRegression(),
    "Lasso Regression": Lasso(alpha=10.0),
    "Ridge Regression": Ridge(alpha=100.0)
}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21)

for name, model in models.items():
    pipe = Pipeline(steps=[
        ('pre', pre),
        ('model', model)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    print(f'{name} RMSE: {rmse:.4f}')
    print("----------------------------------")

Linear Regression RMSE: 11655.7608
----------------------------------
Lasso Regression RMSE: 11601.2516
----------------------------------
Ridge Regression RMSE: 11429.7883
----------------------------------


*Zaključak: sva tri modela imaju vrlo sličan RMSE, no Ridge Regression je najbolji model zbog najniže vrijednosti RMSE.*

#7. ZADATAK
Koristite dataset sa linka koji ste preuzeli prethodno, a link je u datoteci datasets_klasifikacija_p6.txt. Istrenirajte model logističke regresije koristeći GridSearchCV, isprobajte promjene dva hiperparametra navedena ispod i i ispišite koje su točnosti modela, ako zadate 5 foldova u cross_val_score metodi.

log_reg = LogisticRegression(
    C=0.5,          # jačina regularizacije, a tipične vrijednosti su  [0.01, 1, 10]
    penalty='l2',   # tip regularizacije, a može biti 'l1', 'elasticnet', 'none')

Nakon toga istrenirajte model SVC koristeći GridSearchCV, isprobajte promjene dva hiperparametra navedena ispod i ispišite koje su točnosti modela, ako zadate 5 foldova u cross_val_score metodi. Evo hiperparametara u nastavku:

svc = SVC(
    C=1.0,          # jačina regularizacije, a tipične vrijednosti su  [0.01, 1, 10]
    kernel='linear',# linearni kernel, a može biti 'poly', 'rbf', 'sigmoid')

In [18]:
#koristim prvi dataframe
y = df["Quality"]
X = df.drop(["Quality"], axis=1)

log_reg = LogisticRegression(solver="liblinear") # liblinear jer podržava l1 i l2
svc = SVC()

#LogisticRegression
param_grid_lgr = {
    'C': [0.01, 1, 10],
    'penalty': ['l1', 'l2']
}

grid_model_lgr = GridSearchCV(log_reg, return_train_score=True, param_grid=param_grid_lgr, cv=5, scoring='accuracy')
grid_model_lgr.fit(X, y)
print(f'Best score RF: {grid_model_lgr.best_score_* 100:.2f}%')
print("Best Hyperparameters LogReg:", grid_model_lgr.best_params_)
print("-----------------------------------------------------------------")


#SVC
param_grid_svc = {
    'C': [0.01, 1, 10],
    'kernel': ['linear', 'poly', 'rbf', 'sigmoid']
}

grid_model_svc = GridSearchCV(svc, return_train_score=True, param_grid=param_grid_svc, cv=5, scoring='accuracy')
grid_model_svc.fit(X, y)
print(f'Best score RF: {grid_model_svc.best_score_* 100:.2f}%')
print("Best Hyperparameters SVC:", grid_model_svc.best_params_)


Best score RF: 84.70%
Best Hyperparameters LogReg: {'C': 10, 'penalty': 'l1'}
-----------------------------------------------------------------
Best score RF: 98.04%
Best Hyperparameters SVC: {'C': 10, 'kernel': 'rbf'}


*Zaključak: najbolji model je SVC s hiperparametrima {'C': 10, 'kernel': 'rbf'} jer postiže najvišu točnost (98.04%) u usporedbi s Logističkom regresijom koja s {'C': 10, 'penalty': 'l1'} postiže 84.70% u svojoj najboljoj kombinaciji.*

#8. ZADATAK
Koristite dataset sa linka koji ste preuzeli prethodno, a link je u datoteci  datasets_klasifikacija_p6.txt. Istrenirajte ansambl modele Random Forest i Gradient Boost. Standardizacija u ovom slučaju nije potrebna jer ako se ne zada bazni model, koristi se stablo odlučivanja koje ne zahtijeva standardizaciju. Koristite GridSearcCV, a mijenjajte samo jedan hiperparametar po želji (dvije vrijednosti) i ispišite točnosti.

In [19]:
#koristim prvi dataframe
y = df["Quality"]
X = df.drop(["Quality"], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

#GradientBoosting
param_grid_gb = {
    'max_depth': [5, 10]
}

gb = ensemble.GradientBoostingClassifier(random_state=42)
grid_gb = GridSearchCV(gb, return_train_score=True, param_grid=param_grid_gb, cv=5, scoring='accuracy')
grid_gb.fit(X_train, y_train)
print(f'Best score GB: {grid_gb.best_score_ * 100:.2f}%')
print(f'Best params GB: {grid_gb.best_params_}')
print("-----------------------------------------------------------------")


#RandomForest
param_grid_rf = {
    'max_depth': [5, 10]
}

rf = ensemble.RandomForestClassifier(random_state=42)
grid_rf = GridSearchCV(rf, return_train_score=True, param_grid=param_grid_rf, cv=5, scoring='accuracy')
grid_rf.fit(X_train, y_train)
print(f'Best score RF: {grid_rf.best_score_* 100:.2f}%')
print(f'Best params RF: {grid_rf.best_params_}')


Best score GB: 96.77%
Best params GB: {'max_depth': 5}
-----------------------------------------------------------------
Best score RF: 96.75%
Best params RF: {'max_depth': 10}


*Zaključak: Gradient Boosting s hiperparametrom {'max_depth': 5} postiže najvišu točnost (96.77%), dok Random Forest s {'max_depth': 10} postiže 96.75%.*

#9. ZADATAK
Koristite dataset sa linka koji ste preuzeli prethodno, a link je u datoteci datasets_klasifikacija_p6.txt. Istrenirajte SVC modele za kernelima poly i rbf, ali za svaki izaberite po jedan hiperparametar i isprobajte po dvije vrijednosti koristeći GridSearchCV. Ispišite koje koji model je najbolji, odnosno hiperparametre najboljeg modela.

In [28]:
#koristim prvi dataframe
y = df["Quality"]
X = df.drop(["Quality"], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

param_grid_svc_9 = [
    {'kernel': ['poly'], 'degree': [2, 3]},
    {'kernel': ['rbf'], 'gamma': [0.1, 1]}
]
grid_svc_9 = GridSearchCV(svm.SVC(), param_grid_svc_9, cv=5, scoring='accuracy')
grid_svc_9.fit(X_train, y_train)
print(f'Best score SVC: {grid_svc_9.best_score_* 100:.2f}%')
print(f'Best params SVC: {grid_svc_9.best_params_}')


Best score SVC: 98.39%
Best params SVC: {'gamma': 0.1, 'kernel': 'rbf'}


*Zaključak: Od isprobanih hiperparametara za SVC model, najbolja kombinacija je gamma = 0.1 i kernel = 'rbf', pri čemu model postiže najbolju točnost od 98,39%.*

#10. ZADATAK
Koristeći biblioteku ELI5 prikažite ukupan utjecaj pojedinih značajki modela Random Forest koji kreirajte koristeći bolji hiperparametar otkriven u zadatku #8. Osim toga prikažite utjecaj pojedine značajke za prvi primjer u testnom datasetu.

In [54]:
#koristim prvi dataframe
y = df["Quality"]
X = df.drop(["Quality"], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(max_depth=10, random_state=42)
model.fit(X_train, y_train)

#show_weights
feature_names = X_test.columns.tolist()
perm = PermutationImportance(model, random_state=1).fit(X_test, y_test)
display(eli5.show_weights(perm, feature_names=feature_names))
print()
print("---------------------------------")
print()

#show_prediction
i = 0
target_names = ['low', 'high']
eli5.show_prediction(model, X_test.iloc[i], feature_names=feature_names, target_names=target_names)

Weight,Feature
0.1160 ± 0.0088,Weight
0.0980 ± 0.0050,Softness
0.0969 ± 0.0148,HarvestTime
0.0741 ± 0.0079,Sweetness
0.0724 ± 0.0130,Ripeness
0.0703 ± 0.0084,Size
0.0244 ± 0.0052,Acidity



---------------------------------



*Zaključak:*
*na predikciju modela (vjerojatnost od 98,3%) naviše utječu sljedeće značajke: Weight, Softness i HarvestTime te BIAS (osnovna vjerojatnost prije uključivanja navedenih značajki)*